# 🔍 JobFinder Scraper
Scrapes entry-level & new grad job listings from **Arbeitnow** and **RemoteOK** using their free public APIs.

Just run all cells top to bottom!

In [ ]:
# Install dependencies
!pip install requests pandas -q

In [ ]:
import requests
import pandas as pd
from IPython.display import display, HTML

ENTRY_LEVEL_KEYWORDS = [
    "new grad", "new graduate", "recent graduate", "entry level", "entry-level",
    "junior", "0-1 years", "0-2 years", "0-3 years", "1 year of experience",
    "no experience required", "early career", "associate",
]

def matches_entry_level(text):
    text_lower = text.lower()
    return [kw for kw in ENTRY_LEVEL_KEYWORDS if kw in text_lower]

print("✅ Setup complete")

In [ ]:
def scrape_arbeitnow(max_pages=4):
    jobs = []
    seen = set()

    for page in range(1, max_pages + 1):
        try:
            resp = requests.get(
                "https://www.arbeitnow.com/api/job-board-api",
                params={"page": page},
                headers={"Accept": "application/json"},
                timeout=15,
            )
            resp.raise_for_status()
            items = resp.json().get("data", [])
        except Exception as e:
            print(f"[arbeitnow] Page {page} error: {e}")
            break

        if not items:
            break

        for job in items:
            title = job.get("title", "").strip()
            url   = job.get("url", "").strip()
            if not title or not url or url in seen:
                continue
            desc  = job.get("description", "")
            tags  = ", ".join(job.get("tags", []))
            matched = matches_entry_level(f"{title} {desc} {tags}")
            if not matched:
                continue
            seen.add(url)
            jobs.append({
                "title":    title,
                "company":  job.get("company_name", ""),
                "location": job.get("location") or ("Remote" if job.get("remote") else ""),
                "source":   "Arbeitnow",
                "keywords": ", ".join(matched),
                "url":      url,
            })

    print(f"[Arbeitnow] Found {len(jobs)} entry-level jobs")
    return jobs

arbeitnow_jobs = scrape_arbeitnow()

In [ ]:
def scrape_remoteok():
    jobs = []
    seen = set()
    tags = ["junior", "entry-level", "new-grad"]

    for tag in tags:
        try:
            resp = requests.get(
                "https://remoteok.com/api",
                params={"tag": tag},
                headers={"User-Agent": "Mozilla/5.0 (compatible; JobFinder/1.0)"},
                timeout=15,
            )
            resp.raise_for_status()
            data = resp.json()
            if isinstance(data, list) and data:
                data = data[1:]  # skip metadata item
        except Exception as e:
            print(f"[remoteok] Tag '{tag}' error: {e}")
            continue

        for job in data:
            title = job.get("position", "").strip()
            url   = job.get("url", "").strip()
            if not title or not url or url in seen:
                continue
            seen.add(url)
            matched = matches_entry_level(f"{title} {' '.join(job.get('tags', []))}")
            jobs.append({
                "title":    title,
                "company":  job.get("company", ""),
                "location": job.get("location") or "Remote",
                "source":   "RemoteOK",
                "keywords": ", ".join(matched) if matched else tag,
                "url":      url,
            })

    print(f"[RemoteOK] Found {len(jobs)} entry-level jobs")
    return jobs

remoteok_jobs = scrape_remoteok()

In [ ]:
all_jobs = arbeitnow_jobs + remoteok_jobs
df = pd.DataFrame(all_jobs)

print(f"\n📋 Total jobs found: {len(df)}")
print(f"   Arbeitnow: {len(arbeitnow_jobs)}")
print(f"   RemoteOK:  {len(remoteok_jobs)}")

In [ ]:
# Display as a clickable table
def make_link(url, text="Apply"):
    return f'<a href="{url}" target="_blank">{text}</a>'

display_df = df.copy()
display_df["link"] = display_df["url"].apply(make_link)
display_df = display_df.drop(columns=["url"])

display(HTML(display_df.to_html(escape=False, index=False)))

In [ ]:
# Optional: export to CSV
df.to_csv("jobs.csv", index=False)
print("✅ Saved to jobs.csv")

# Download it
from google.colab import files
files.download("jobs.csv")